[![Homepage](https://img.shields.io/badge/homepage-blueviolet?logo=htmx)](https://www.bendai.org/CUHK-STAT3009/)
[![GitHub](https://img.shields.io/badge/GitHub-black.svg?logo=github)](https://github.com/statmlben/CUHK-STAT3009)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statmlben/CUHK-STAT3009/blob/main/STAT3009_nb2026/STAT3009_ML_I.ipynb)

# Notebook 03: Machine Learning I — Models and estimators

**STAT3009 · Recommender Systems**  
**Suggested time:** 75–90 minutes

The warm-up reviewed how raw recommender IDs become `X_train`, `y_train`, and `X_test`. This notebook starts from those arrays and develops a general language for prediction models.

```text
examples → choose a model family → fit parameters → predict new rows
         → inspect fitted state → evaluate predictions
```

By the end, you should be able to:

1. explain the roles of a model, parameters, loss, fitting, and prediction;
2. use `LinearRegression` through its constructor, `fit`, and `predict`;
3. distinguish constructor parameters from fitted attributes;
4. implement recommender baselines as sklearn-style estimators;
5. evaluate different estimators with one shared workflow.

## 1. Setup

We use NumPy and pandas for data work. Scikit-learn supplies a regression model, evaluation metric, and the base classes used by our custom estimators.

In [ ]:
import sys

import numpy as np
import pandas as pd
import sklearn

from IPython.display import display
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.exceptions import NotFittedError
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.validation import check_is_fitted

pd.set_option('display.max_rows', 12)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)

## 2. From a model family to a fitted model

We begin with a tiny regression problem whose answer is easy to verify. Each row of `X_train_small` is one example with two features. The entry at the same position in `y_train_small` is its target.

In [ ]:
X_train_small = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [2.0, 1.0],
])

y_train_small = np.array([3.0, 2.0, 4.0, 6.0])

small_data = pd.DataFrame(X_train_small, columns=['x1', 'x2'])
small_data['y'] = y_train_small
display(small_data)
print('X shape:', X_train_small.shape)
print('y shape:', y_train_small.shape)

### Model family and training loss

Consider the linear model family

$$\widehat y = \theta_0 + \theta_1 x_1 + \theta_2 x_2.$$

A parameter setting $(\theta_0,\theta_1,\theta_2)$ produces one candidate prediction rule. We will calculate the training loss for one candidate step by step.

In [ ]:
# Candidate C: y_hat = 1 + 1.5 x1 + 1.2 x2
candidate_C_predictions = (
    1.0
    + 1.5 * X_train_small[:, 0]
    + 1.2 * X_train_small[:, 1]
)

candidate_C_squared_errors = (
    y_train_small - candidate_C_predictions
) ** 2
candidate_C_mse = candidate_C_squared_errors.mean()

print('observed y:    ', y_train_small)
print('predicted y:   ', candidate_C_predictions)
print('squared errors:', candidate_C_squared_errors)
print('training MSE:  ', candidate_C_mse)

Repeating the same calculation for three parameter settings gives:

| candidate | $(\theta_0,\theta_1,\theta_2)$ | training MSE |
|---|---:|---:|
| A | $(0,1,1)$ | $4.500$ |
| B | $(1,2,1)$ | $0.000$ |
| C | $(1,1.5,1.2)$ | $0.255$ |

Training searches for a parameter setting with small average loss:

$$\widehat\theta = \arg\min_{\theta}\frac{1}{n}\sum_{j=1}^{n}\left(y_j-f_{\theta}(x_j)\right)^2.$$

The **minimum** is the smallest loss value. The **arg minimum** is the parameter setting that produces it.

### Meet `LinearRegression`

`LinearRegression` is a class provided by `sklearn.linear_model`. It represents the linear model family

$$\widehat y=\theta_0+\theta_1x_1+\theta_2x_2.
$$

Read the next line in three parts:

```python
model_small = LinearRegression(fit_intercept=True)
# variable       class             option
```

- `LinearRegression(...)` creates a new Python model object;
- `fit_intercept=True` allows the model to learn the intercept $\theta_0$;
- `model_small` is the variable that refers to this object.

At this point the object has its settings, but it has not seen `X_train_small` or `y_train_small`, so no coefficients have been learned. The following `fit(...)` call will add that learned state.

In [ ]:
model_small = LinearRegression(fit_intercept=True)

print('object type:', type(model_small).__name__)
print('constructor parameter:', model_small.get_params()['fit_intercept'])
print('has coef_ before fit:', hasattr(model_small, 'coef_'))
print('object state before fit:', model_small.__dict__)

In [ ]:
returned_object = model_small.fit(X_train_small, y_train_small)

print('fit returned the same object:', returned_object is model_small)
print('coef_:', model_small.coef_)
print('intercept_:', model_small.intercept_)
print('n_features_in_:', model_small.n_features_in_)

`fit_intercept` is a **constructor parameter** chosen before training. `coef_` and `intercept_` are **fitted attributes** created from the training data. The trailing underscore marks learned public state in scikit-learn.

In [ ]:
X_new_small = np.array([
    [0.0, 2.0],
    [3.0, 0.0],
])

y_pred_small = model_small.predict(X_new_small)
y_pred_manual = (
    model_small.intercept_
    + X_new_small @ model_small.coef_
)

prediction_check = pd.DataFrame({
    'x1': X_new_small[:, 0],
    'x2': X_new_small[:, 1],
    'model.predict': y_pred_small,
    'manual calculation': y_pred_manual,
})

display(prediction_check)
assert np.allclose(y_pred_small, y_pred_manual)

### Checkpoint

1. Which method receives both `X` and `y`?
2. Which method receives new feature rows only?
3. Why does `predict(X_new)` preserve the row order?
4. Which values existed before fitting, and which appeared after fitting?

## 3. A complete regression example: California housing

The course repository provides a fixed train/test split of the California housing dataset. The target `MedHouseVal` is measured in units of $100{,}000$.

Dataset documentation: [scikit-learn California housing data](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html).

In [ ]:
HOUSING_BASE_URL = (
    'https://raw.githubusercontent.com/'
    'statmlben/CUHK-STAT3009/main/dataset/housing'
)

housing_train = pd.read_csv(f'{HOUSING_BASE_URL}/train.csv')
housing_test = pd.read_csv(f'{HOUSING_BASE_URL}/test.csv')

print('train shape:', housing_train.shape)
print('test shape: ', housing_test.shape)
display(housing_train.head())

In [ ]:
feature_columns = [
    'MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
    'Population', 'AveOccup', 'Latitude', 'Longitude',
]
target_column = 'MedHouseVal'

X_train_housing = housing_train[feature_columns].to_numpy(dtype=float)
y_train_housing = housing_train[target_column].to_numpy(dtype=float)
X_test_housing = housing_test[feature_columns].to_numpy(dtype=float)
y_test_housing = housing_test[target_column].to_numpy(dtype=float)

print('X_train:', X_train_housing.shape)
print('y_train:', y_train_housing.shape)
print('X_test: ', X_test_housing.shape)
print('y_test: ', y_test_housing.shape)

assert X_train_housing.shape[1] == len(feature_columns)
assert len(X_train_housing) == len(y_train_housing)

The data contract is the same as in the small example:

- row `j` of `X_train_housing` describes one block group;
- position `j` of `y_train_housing` stores its known target;
- `predict(X_test_housing)` must return one value per test row.

In [ ]:
housing_model = LinearRegression()
housing_model.fit(X_train_housing, y_train_housing)

y_pred_housing = housing_model.predict(X_test_housing)
housing_rmse = root_mean_squared_error(
    y_test_housing, y_pred_housing
)

print('prediction shape:', y_pred_housing.shape)
print(f'test RMSE: {housing_rmse:.3f}')
assert y_pred_housing.shape == y_test_housing.shape

The fitted equation is available through `coef_` and `intercept_`. Coefficient magnitude alone is not a fair feature-importance comparison when columns use different units, and the fitted association is not automatically causal.

## 4. Read an estimator as a Python object

A scikit-learn estimator moves through two states:

```text
constructor → configured estimator → fit(X, y) → fitted estimator
```

The same object remains in memory. Fitting adds learned attributes.

In [ ]:
state_demo = LinearRegression(fit_intercept=True)
state_before_fit = state_demo.__dict__.copy()

returned_object = state_demo.fit(
    X_train_small, y_train_small
)
state_after_fit = state_demo.__dict__.copy()

print('state before fit:')
display(state_before_fit)
print('state after fit:')
display(state_after_fit)
print('fit returned self:', returned_object is state_demo)

| Term | Chosen or learned? | Example |
|---|---|---|
| constructor parameter | chosen before `fit` | `fit_intercept=True` |
| learned model parameter | estimated from training examples | $\widehat\theta$ |
| fitted attribute | Python storage for learned state | `coef_`, `intercept_` |

The word *parameter* is overloaded. Mathematical parameters are learned, while sklearn often uses *parameter* for constructor configuration.

In [ ]:
print('constructor parameters:')
display(state_demo.get_params())

print('selected fitted attributes:')
display({
    'coef_': state_demo.coef_,
    'intercept_': state_demo.intercept_,
    'n_features_in_': state_demo.n_features_in_,
})

### Change configuration with `set_params`

`set_params` changes constructor configuration and returns the same estimator object. Fit the estimator after changing its configuration. If an already fitted estimator receives new settings, fit it again before making predictions.

In [ ]:
configured_model = LinearRegression()
returned_object = configured_model.set_params(
    fit_intercept=False
)

print('set_params returned self:', returned_object is configured_model)
print('fit_intercept:', configured_model.get_params()['fit_intercept'])
print('has coef_ before fit:', hasattr(configured_model, 'coef_'))

configured_model.fit(X_train_small, y_train_small)
print('intercept_ after fit:', configured_model.intercept_)

### Checkpoint

For each name below, classify it as a constructor parameter, method, or fitted attribute:

```text
fit_intercept    fit    predict    coef_    intercept_
```

## 5. Return to Netflix rating prediction

The warm-up explained the preprocessing in detail. Here we repeat it compactly so that this notebook can run independently.

The course split makes the test user–movie pairs available as model inputs. We may use their **ID columns** to define a consistent vocabulary. Their **ratings** remain hidden from `fit`.

### Classroom evaluation and the competition test

The published `test.csv` used in this notebook is a **labeled classroom evaluation split**. We keep its input pairs in `X_test` and name its labels `y_test_demo`. Cells that use `y_test_demo` demonstrate final evaluation only.

In the course competition, students receive `X_test`, but the real `y_test` remains hidden. A test score must not determine which model to try next. ML II will introduce validation data for model comparison.

In [ ]:
NETFLIX_BASE_URL = (
    'https://raw.githubusercontent.com/'
    'statmlben/CUHK-STAT3009/main/dataset/netflix'
)

ID_TYPES = {'user_id': 'string', 'movie_id': 'string'}
COLUMNS = ['user_id', 'movie_id', 'rating']

train_raw = pd.read_csv(
    f'{NETFLIX_BASE_URL}/train.csv',
    usecols=COLUMNS, dtype=ID_TYPES,
)[COLUMNS]
test_demo_raw = pd.read_csv(
    f'{NETFLIX_BASE_URL}/test.csv',
    usecols=COLUMNS, dtype=ID_TYPES,
)[COLUMNS]

print('train shape:', train_raw.shape)
print('classroom test shape:', test_demo_raw.shape)
display(train_raw.head())

In [ ]:
all_ids = pd.concat(
    [
        train_raw[['user_id', 'movie_id']],
        test_demo_raw[['user_id', 'movie_id']],
    ],
    ignore_index=True,
)

user_encoder = LabelEncoder().fit(all_ids['user_id'])
item_encoder = LabelEncoder().fit(all_ids['movie_id'])

train = train_raw.copy()
test_demo = test_demo_raw.copy()

train['user_id'] = user_encoder.transform(train_raw['user_id'])
test_demo['user_id'] = user_encoder.transform(test_demo_raw['user_id'])
train['movie_id'] = item_encoder.transform(train_raw['movie_id'])
test_demo['movie_id'] = item_encoder.transform(test_demo_raw['movie_id'])

In [ ]:
X_train = train[['user_id', 'movie_id']].to_numpy(dtype=int)
y_train = train['rating'].to_numpy(dtype=float)
X_test = test_demo[['user_id', 'movie_id']].to_numpy(dtype=int)
y_test_demo = test_demo['rating'].to_numpy(dtype=float)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test: ', X_test.shape)
print('y_test_demo:', y_test_demo.shape)

assert X_train.shape[1] == 2
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test_demo)

## 6. From the global-mean specification to `GlobalMeanRS`

Before reading the class, answer the same four questions:

| Module | Global mean |
|---|---|
| model | $f(u,i)=\mu$: every pair receives one constant |
| learned parameter | $\widehat\mu=\frac{1}{\lvert\mathcal R_{tr}\rvert}\sum_{(u,i,r)\in\mathcal R_{tr}}r$ |
| hyperparameters | none |
| loss / criterion | $J(\mu)=\sum_{(u,i,r)\in\mathcal R_{tr}}(r-\mu)^2$ |

The training mean is the constant that minimizes squared error. Now translate the specification into methods:

- `fit(X, y)` calculates $\widehat\mu$, stores it as `global_mean_`, and returns `self`;
- `predict(X_new)` returns that fitted value once per input row;
- `check_is_fitted` prevents prediction before `global_mean_` exists.

This model ignores the values inside each user–movie pair, but `X_new` still determines how many predictions to return.

In [ ]:
class GlobalMeanRS(RegressorMixin, BaseEstimator):
    def fit(self, X, y):
        self.global_mean_ = np.asarray(y).mean()
        return self

    def predict(self, X):
        check_is_fitted(self, 'global_mean_')
        return np.full(len(X), self.global_mean_)

### Prediction before fitting

The estimator does not contain `global_mean_` immediately after construction. `check_is_fitted` converts that missing state into a clear `NotFittedError`.

In [ ]:
unfitted_model = GlobalMeanRS()

try:
    unfitted_model.predict(X_test[:3])
except NotFittedError as error:
    print(type(error).__name__)
    print(str(error).splitlines()[0])

In [ ]:
global_model = GlobalMeanRS()
print('state before fit:', global_model.__dict__)

global_model.fit(X_train, y_train)
y_pred_global = global_model.predict(X_test)
rmse_global_demo = root_mean_squared_error(
    y_test_demo, y_pred_global
)

print('state after fit:', global_model.__dict__)
print('prediction shape:', y_pred_global.shape)
print(f'classroom evaluation RMSE: {rmse_global_demo:.3f}')

## 7. From the user-mean specification to `UserMeanRS`

Start with the mathematics, then read the implementation:

| Module | User mean |
|---|---|
| model | $f(u,i)=a_u$; use $\mu$ when user $u$ was absent from training |
| learned parameters | $\widehat a_u=\frac{1}{\lvert\mathcal I_u^{tr}\rvert}\sum_{i\in\mathcal I_u^{tr}}r_{ui}$ and fallback $\widehat\mu$ |
| hyperparameters | none |
| loss / criterion | $J_u(a)=\sum_{i\in\mathcal I_u^{tr}}(r_{ui}-a)^2$ for each user |

For every observed user, the user mean minimizes that user's squared errors. The methods now have clear jobs:

- `fit(X, y)` groups ratings by user and stores `global_mean_` and `user_means_`;
- `predict(X_new)` looks up the user column and preserves the global fallback for an unseen user.

In [ ]:
class UserMeanRS(RegressorMixin, BaseEstimator):
    def fit(self, X, y):
        X, y = np.asarray(X), np.asarray(y)
        users = X[:, 0].astype(int)

        self.global_mean_ = y.mean()
        self.user_means_ = np.full(
            users.max() + 1, self.global_mean_
        )

        for user in set(users):
            ratings = y[users == user]
            self.user_means_[user] = ratings.mean()

        return self

    def predict(self, X):
        check_is_fitted(
            self, ['global_mean_', 'user_means_']
        )
        users = np.asarray(X)[:, 0].astype(int)

        predictions = np.full(len(users), self.global_mean_)
        valid = (users >= 0) & (users < len(self.user_means_))
        predictions[valid] = self.user_means_[users[valid]]
        return predictions

In [ ]:
user_model = UserMeanRS().fit(X_train, y_train)

print('global_mean_:', user_model.global_mean_)
print('user_means_ shape:', user_model.user_means_.shape)
print('first three entries:', user_model.user_means_[:3])

### Known user and unseen user

The item column remains part of the estimator input contract, although this particular model uses only column 0.

In [ ]:
train_user_ids = np.unique(X_train[:, 0])
test_user_ids = np.unique(X_test[:, 0])
test_only_user_ids = np.setdiff1d(test_user_ids, train_user_ids)

known_user = int(train_user_ids[0])
if len(test_only_user_ids) > 0:
    unseen_user = int(test_only_user_ids[0])
else:
    unseen_user = int(max(test_user_ids.max(), train_user_ids.max()) + 1)

example_item = int(X_test[0, 1])
X_demo = np.array([
    [known_user, example_item],
    [unseen_user, example_item],
])
demo_predictions = user_model.predict(X_demo)

prediction_demo = pd.DataFrame({
    'user status': ['seen in training', 'absent from training'],
    'encoded user': X_demo[:, 0],
    'prediction': demo_predictions,
    'source': ['user mean', 'global-mean fallback'],
})
display(prediction_demo)

assert np.isclose(
    demo_predictions[0], user_model.user_means_[known_user]
)
assert np.isclose(
    demo_predictions[1], user_model.global_mean_
)

In [ ]:
y_pred_user = user_model.predict(X_test)
rmse_user_demo = root_mean_squared_error(
    y_test_demo, y_pred_user
)

print('prediction shape:', y_pred_user.shape)
print(f'classroom evaluation RMSE: {rmse_user_demo:.3f}')

## 8. One evaluation workflow for every estimator

The workflow below depends only on the shared estimator interface. It does not need model-specific `if` statements.

In [ ]:
models = [
    GlobalMeanRS(),
    UserMeanRS(),
]

evaluation_rows = []

for model in models:
    fitted_model = model.fit(X_train, y_train)
    predictions = fitted_model.predict(X_test)

    evaluation_rows.append({
        'estimator': type(model).__name__,
        'prediction_shape': predictions.shape,
        'demo_RMSE': root_mean_squared_error(
            y_test_demo, predictions
        ),
    })

evaluation_results = pd.DataFrame(evaluation_rows)
display(evaluation_results.sort_values('demo_RMSE'))

### Metrics stay outside the estimator

`RegressorMixin` provides a default `.score(X, y)` method, but its value is $R^2$, not RMSE. The course therefore calls `root_mean_squared_error` explicitly. Larger $R^2$ is better, while smaller RMSE is better.

In [ ]:
user_model = UserMeanRS().fit(X_train, y_train)
user_predictions = user_model.predict(X_test)

print(f'R² from model.score: {user_model.score(X_test, y_test_demo):.3f}')
print(
    'RMSE from explicit metric:',
    f'{root_mean_squared_error(y_test_demo, user_predictions):.3f}',
)

### Bridge to ML II: how should we compare models?

In our competition setting, `X_test` is available because we must produce one prediction for every test user–movie pair. The real values in `y_test` are the hidden answers.

Cells tagged `instructor-evaluation` use the separate variable `y_test_demo` from the published classroom split. They reveal evaluation only after the example models have been fixed. During model development, we should not repeatedly choose the next model from this score. That would make the evaluation labels part of model selection.

**Question for ML II:** if `y_test` stays hidden, which data should compare candidate estimators? Holdout validation and cross-validation provide the answer.

## 9. In-class practice: from item mean to `ItemMeanRS`

Specify the method before writing the class:

| Module | Item mean |
|---|---|
| model | $f(u,i)=b_i$; use $\mu$ when item $i$ was absent from training |
| learned parameters | $\widehat b_i=\frac{1}{\lvert\mathcal U_i^{tr}\rvert}\sum_{u\in\mathcal U_i^{tr}}r_{ui}$ and fallback $\widehat\mu$ |
| hyperparameters | none |
| loss / criterion | $J_i(b)=\sum_{u\in\mathcal U_i^{tr}}(r_{ui}-b)^2$ for each item |

Now adapt `UserMeanRS` so that it implements this specification. Your estimator should:

1. read encoded movie IDs from column 1;
2. store `global_mean_` and a one-dimensional NumPy array named `item_means_` during `fit`;
3. return the global mean for a movie absent from training;
4. follow the same `fit(X, y)` and `predict(X_new)` contract.

In [ ]:
# class ItemMeanRS(RegressorMixin, BaseEstimator):
#     def fit(self, X, y):
#         # Your code here
#         return self
#
#     def predict(self, X):
#         # Your code here
#         ...

### Solution

In [ ]:
class ItemMeanRS(RegressorMixin, BaseEstimator):
    def fit(self, X, y):
        X, y = np.asarray(X), np.asarray(y)
        items = X[:, 1].astype(int)

        self.global_mean_ = y.mean()
        self.item_means_ = np.full(
            items.max() + 1, self.global_mean_
        )

        for item in set(items):
            ratings = y[items == item]
            self.item_means_[item] = ratings.mean()

        return self

    def predict(self, X):
        check_is_fitted(
            self, ['global_mean_', 'item_means_']
        )
        items = np.asarray(X)[:, 1].astype(int)

        predictions = np.full(len(items), self.global_mean_)
        valid = (items >= 0) & (items < len(self.item_means_))
        predictions[valid] = self.item_means_[items[valid]]
        return predictions

In [ ]:
models = [
    GlobalMeanRS(),
    UserMeanRS(),
    ItemMeanRS(),
]

final_rows = []

for model in models:
    predictions = model.fit(X_train, y_train).predict(X_test)
    final_rows.append({
        'estimator': type(model).__name__,
        'demo_RMSE': root_mean_squared_error(
            y_test_demo, predictions
        ),
    })

final_results = pd.DataFrame(final_rows)
display(final_results.sort_values('demo_RMSE'))

## Exit ticket

Answer each question in one or two sentences:

1. What is the difference between a model family and a fitted model?
2. Why does `fit` receive `y`, while `predict` does not?
3. What is the difference between `fit_intercept` and `coef_`?
4. Why should learned public attributes end in `_`?
5. How can one evaluation loop work for several different estimators?
6. Why do we call RMSE explicitly instead of relying on `.score(...)`?
7. Complete the coding cell below without copying one of the earlier estimator classes.

### Coding exit ticket

Complete a constant-rating estimator. `fit` should learn one public fitted attribute named `constant_`; `predict` should check the fitted state and return one value per input row.

In [ ]:
class ConstantRS(RegressorMixin, BaseEstimator):
    def fit(self, X, y):
        # Learn self.constant_ from y.
        ...
        # Return the same estimator object.
        ...

    def predict(self, X):
        # Check that constant_ exists.
        ...
        # Return an array with len(X) predictions.
        ...

## Summary

```text
constructor parameters
        ↓
configured estimator
        ↓ fit(X_train, y_train)
fitted attributes
        ↓ predict(X_new)
prediction vector
        ↓ explicit metric
generalization performance
```

The algorithms differ, but the estimator contract remains the same. ML II will use this common interface to compare candidate models without repeatedly consulting the final test labels.